# Lesson 03 - Агентни дизайнерски модели

В този урок разглеждаме три основни дизайнерски модела за изграждане на ефективни AI агенти:

1. **Ясни инструкции за агента** — Създаване на прецизни, ролеви указания, които насочват поведението на агента  
2. **Структуриран изход с Pydantic модели** — Осигуряване на предвидими, валидирани данни от агента  
3. **Агенти с единствена отговорност** — Проектиране на фокусирани агенти, които изпълняват една задача отлично

Ще приложим всеки модел към сценарий с **препоръчител на туристически дестинации**, като последователно изграждаме система, която може да предлага дестинации, да проверява наличности и да управлява логистиката.


## Настройка


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Шаблон 1: Ясни инструкции за агента

Най-въздействащият шаблон е и най-простият: написването на ясни, подробни инструкции за вашия агент.

Добри инструкции определят:
- **Кой** е агентът (персона и тон)
- **Какво** трябва да прави (стъпка по стъпка отговорности)
- **Как** трябва да се държи (ограничения и стил)

По-долу създаваме агент-консиерж за пътуване с изрични инструкции, които оформят всеки отговор, който той произвежда.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Модел 2: Структуриран изход с Pydantic модели

Свободният текст е полезен за разговор, но системите надолу по веригата се нуждаят от структурирани данни.  
Комбинирайки **Pydantic модели** с **функция инструмент**, можем:

- Да определим точна схема за изхода на агента  
- Автоматично да валидираме отговорите  
- Да интегрираме резултатите от агента в логиката на приложението по надежден начин  

Също така въвеждаме инструмент, който връща подробности за дестинацията, така че агентът да основава препоръките си на реални данни.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Модел 3: Агенти с единствена отговорност

Сложните задачи се подобряват чрез разделяне на работата между няколко фокусирани агенти, всеки със своя единствена отговорност:

- **Експерт по дестинации**, който знае за места и наличности
- **Планиращ логистика**, който се занимава с полети, хотели и маршрути

Това отразява принципа на софтуерното инженерство за *разделяне на отговорностите* — всеки агент е по-лесен за тестване, поддръжка и усъвършенстване независимо.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Обобщение

В този урок приложихме три агентски дизайн шаблона към сценарио за препоръчване на пътувания:

| Шаблон | Основна идея | Полза |
|---|---|---|
| **Ясни инструкции** | Определете персона, отговорности и ограничения предварително | Последователно, съобразено с бранда поведение на агента |
| **Структуриран изход** | Използвайте Pydantic модели като формат на отговора | Валидация, резултати достъпни за машинна обработка |
| **Единична отговорност** | Дайте на всеки агент една фокусирана задача | По-лесно за тестване, поддръжка и съставяне |

Тези шаблони се съчетават естествено — можете да комбинирате ясни инструкции със структуриран изход в агент с единична отговорност, за да изградите стабилни, готови за продукция системи.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
